In [ ]:
# @title 1. Install & Update Libraries
# CRITICAL: We update transformers to fix the "KeyError" with Phi-3
!pip install -q -U "transformers>=4.41.2" accelerate bitsandbytes langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pdfplumber
!pip install -q sentence-transformers
print("-" * 50)
print("✅ LIBRARIES INSTALLED")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 26.6 MB/s eta 0:00:00
ERROR: Operation cancelled by user
ERROR: Operation cancelled by user
--------------------------------------------------
✅ LIBRARIES INSTALLED


In [ ]:
!pip install rouge-score bert-score nltk pandas

In [ ]:
import os
import csv
import time
import torch
import gc
import re
import string
import collections
import pdfplumber
import warnings
import pandas as pd
from datetime import datetime
from typing import List

from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import CrossEncoder, SentenceTransformer, util

# --- NEW METRIC LIBRARIES ---
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# ==========================================
# 1. TABLE-AWARE PDF LOADER (WITH TXT EXPORT)
# ==========================================
def load_pdf_with_tables(path, output_filename="extracted_pdf_data.txt"):
    print(f"📂 Loading {path}...")
    documents = []

    if not os.path.exists(path):
        raise FileNotFoundError(f"File {path} not found. Please upload it to Colab files.")

    with pdfplumber.open(path) as pdf:
        total_pages = len(pdf.pages)
        for i, page in enumerate(pdf.pages):
            if i % 10 == 0: print(f"Processing page {i}/{total_pages}...")

            # Extract Tables
            tables = page.extract_tables()
            table_str = ""
            if tables:
                for table in tables:
                    if not table: continue
                    table_str += "\n\n**[TABLE DATA]**\n"
                    for row in table:
                        clean_row = [str(cell).replace("\n", " ") if cell else "" for cell in row]
                        table_str += "| " + " | ".join(clean_row) + " |\n"

            # Extract Text
            text = page.extract_text() or ""
            full_content = text + "\n" + table_str
            documents.append(Document(page_content=full_content, metadata={"page": i+1}))

    # --- Save to text file for the research paper ---
    print(f"💾 Saving extracted data to {output_filename}...")
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("=== RAG PIPELINE EXTRACTED DATA ===\n")
        f.write(f"Source: {path}\n")
        f.write("====================================\n\n")
        for doc in documents:
            f.write(f"--- PAGE {doc.metadata['page']} ---\n")
            f.write(doc.page_content + "\n\n")

    print(f"✅ Loaded {len(documents)} pages and saved extraction to {output_filename}.")
    return documents

# ==========================================
# 2. CUSTOM MEDCPT EMBEDDINGS (Bi-Encoder)
# ==========================================
class MedCPTEmbeddings(Embeddings):
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"⚙️ Initializing MedCPT on {self.device}...")

        self.article_tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
        self.article_model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder").to(self.device)

        self.query_tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
        self.query_model = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder").to(self.device)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        batch_size = 16
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            with torch.no_grad():
                encoded = self.article_tokenizer(batch, max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
                output = self.article_model(**encoded)
                embeddings = output.last_hidden_state[:, 0, :]
                all_embeddings.extend(embeddings.cpu().tolist())
                del encoded, output, embeddings
                torch.cuda.empty_cache()
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        with torch.no_grad():
            encoded = self.query_tokenizer([text], max_length=512, padding=True, truncation=True, return_tensors="pt").to(self.device)
            output = self.query_model(**encoded)
            return output.last_hidden_state[:, 0, :][0].cpu().tolist()

    def free_article_encoder(self):
        print("🧹 Optimizing Memory: Removing Article Encoder...")
        del self.article_model
        del self.article_tokenizer
        torch.cuda.empty_cache()
        gc.collect()

# ==========================================
# 3. PIPELINE SETUP (FETCHING THE PDF)
# ==========================================
pdf_path = "/content/2024ESC-compressed.pdf"

raw_docs = load_pdf_with_tables(pdf_path, output_filename="extracted_pdf_data.txt")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(raw_docs)

medcpt = MedCPTEmbeddings()
print("🧠 Embedding Documents...")
vectorstore = FAISS.from_documents(splits, medcpt)
medcpt.free_article_encoder()

# ==========================================
# 4. LOAD RERANKER, METRICS MODEL & LLM
# ==========================================
print("⚖️ Loading Cross-Encoder (Reranker)...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')

print("⚙️ Loading Semantic Similarity Model...")
metric_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

print("🚀 Loading Phi-3 Mini...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=False,
    device_map="auto"
)

# ==========================================
# 5. METRICS EVALUATION FUNCTIONS
# ==========================================
def normalize_text(s):
    if pd.isna(s): return ""
    s = str(s).lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def calc_accuracy_metrics(pred, truth):
    norm_pred = normalize_text(pred).split()
    norm_truth = normalize_text(truth).split()

    exact_match = int(norm_pred == norm_truth)

    common = collections.Counter(norm_pred) & collections.Counter(norm_truth)
    num_same = sum(common.values())

    if len(norm_pred) == 0 or len(norm_truth) == 0:
        f1 = precision = recall = int(norm_pred == norm_truth)
    elif num_same == 0:
        f1 = precision = recall = 0.0
    else:
        precision = num_same / len(norm_pred)
        recall = num_same / len(norm_truth)
        f1 = (2 * precision * recall) / (precision + recall)

    return exact_match, round(precision, 4), round(recall, 4), round(f1, 4)

def calc_lexical_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0, 0, 0, 0, 0, 0
    smoothie = SmoothingFunction().method4
    ref = [normalize_text(truth).split()]
    hyp = normalize_text(pred).split()
    b1 = sentence_bleu(ref, hyp, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    b2 = sentence_bleu(ref, hyp, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    b4 = sentence_bleu(ref, hyp, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(str(truth), str(pred))

    return round(b1,4), round(b2,4), round(b4,4), \
           round(rouge_scores['rouge1'].fmeasure,4), \
           round(rouge_scores['rouge2'].fmeasure,4), \
           round(rouge_scores['rougeL'].fmeasure,4)

def calc_semantic_metrics(pred, truth):
    if pd.isna(truth) or not truth: return 0,0,0,0
    P, R, F1 = bert_score([str(pred)], [str(truth)], lang="en", model_type="distilbert-base-uncased", device="cpu")

    embeddings = metric_model.encode([str(pred), str(truth)], convert_to_tensor=True)
    cosine_sim = float(util.pytorch_cos_sim(embeddings[0], embeddings[1])[0][0])

    return round(P.item(),4), round(R.item(),4), round(F1.item(),4), round(cosine_sim,4)

def calc_rag_metrics_llm(query, context, response, expected):
    """LLM-as-a-judge using strict chat templates to prevent prompt confusion."""
    messages = [
        {
            "role": "user",
            "content": (
                "You are a strict grading assistant. Grade the Response against the Context and Expected Answer.\n\n"
                f"Context: {context[:1500]}\n" # Trimmed to prevent overwhelming the judge
                f"Query: {query}\n"
                f"Expected Answer: {expected}\n"
                f"Response: {response}\n\n"
                "Output exactly three numbers from 0 to 10 in this exact format: [FTH, REL, CTX]\n"
                "- FTH (Faithfulness): Is the Response supported by the Context?\n"
                "- REL (Relevancy): Does the Response answer the Query?\n"
                "- CTX (Recall): Is the Expected Answer in the Context?\n"
                "Do NOT output any other text or explanations. Only the brackets and numbers."
            )
        }
    ]

    # Use strict chat template instead of raw pipeline string
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=25,
            do_sample=False, # Force deterministic output
            repetition_penalty=1.0 # Removed penalty to prevent formatting breaks
        )

    input_length = input_ids.shape[1]
    output = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()

    del input_ids, output_ids
    torch.cuda.empty_cache()

    print(f"      [Phi-3 Judge Raw Output]: {output}")

    try:
        numbers = re.findall(r'\d+', output)
        if len(numbers) >= 3:
            fth = min(int(numbers[0]), 10)
            rel = min(int(numbers[1]), 10)
            ctx = min(int(numbers[2]), 10)
            return fth, rel, ctx
    except Exception:
        pass

    return 0, 0, 0

# ==========================================
# 6. CORE PROCESSING FUNCTION
# ==========================================
def process_query(query, expected):
    """Run the full RAG pipeline for a single query and return all metrics."""
    print("🔍 Deep Searching...")

    retrieval_start_time = time.time()
    retrieved_docs = vectorstore.similarity_search(query, k=60)

    unique_docs = []
    seen_pages = set()
    for doc in retrieved_docs:
        page = doc.metadata.get('page')
        if page not in seen_pages:
            unique_docs.append(doc)
            seen_pages.add(page)

    pairs = [[query, doc.page_content] for doc in unique_docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(unique_docs, scores), key=lambda x: x[1], reverse=True)

    top_docs = [doc for doc, score in scored_docs[:5]]
    retrieval_time = round(time.time() - retrieval_start_time, 4)

    context_text = "\n\n".join([d.page_content for d in top_docs])
    source_pages = ", ".join([str(d.metadata.get('page')) for d in top_docs])

    # Use Chat Template for main generation
    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict clinical assistant for the ESC Guidelines. Answer based ONLY on the context. "
                "CRITICAL RULES:\n"
                "1. Define scoring systems strictly.\n"
                "2. State clearly if a treatment is Class III.\n"
                "3. Do not hallucinate acronym meanings.\n"
                "4. If the prompt is not medical, output: 'This model only answers to medical questions.'"
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context_text}\n\nQuestion:\n{query}"
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print("🤖 Synthesizing Answer...")

    generation_start_time = time.time()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=300,
            do_sample=False,    # Greedy decoding to stop word vomit
            repetition_penalty=1.05, # Fixed repetition penalty
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generation_time = time.time() - generation_start_time

    input_length = input_ids.shape[1]
    generated_tokens = len(output_ids[0]) - input_length
    tokens_per_sec = generated_tokens / generation_time if generation_time > 0 else 0

    clean_response = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()

    del input_ids, output_ids
    torch.cuda.empty_cache()

    exact_match, tok_p, tok_r, tok_f1 = calc_accuracy_metrics(clean_response, expected)
    bleu1, bleu2, bleu4, rouge1, rouge2, rougeL = calc_lexical_metrics(clean_response, expected)
    bert_p, bert_r, bert_f1, cos_sim = calc_semantic_metrics(clean_response, expected)
    faithfulness, answer_rel, ctx_recall = calc_rag_metrics_llm(query, context_text, clean_response, expected)

    return {
        "response": clean_response,
        "source_pages": source_pages,
        "exact_match": exact_match,
        "tok_f1": tok_f1, "tok_p": tok_p, "tok_r": tok_r,
        "bleu1": bleu1, "bleu2": bleu2, "bleu4": bleu4,
        "rouge1": rouge1, "rouge2": rouge2, "rougeL": rougeL,
        "bert_f1": bert_f1, "bert_p": bert_p, "bert_r": bert_r,
        "cos_sim": cos_sim,
        "faithfulness": faithfulness,
        "answer_rel": answer_rel,
        "ctx_recall": ctx_recall,
        "retrieval_time": retrieval_time,
        "generation_time": round(generation_time, 2),
        "tokens_per_sec": round(tokens_per_sec, 2),
        "scored_docs": scored_docs
    }

# ==========================================
# 7. AUTOMATED TEST SUITE WITH FULL METRICS
# ==========================================
input_csv_path = "RAG Test Prompts for Medical Guidelines - RAG Test Prompts for Medical Guidelines.csv"
output_csv_path = "rag_evaluation_phi3_full_metrics.csv"

try:
    print(f"📂 Loading prompts from {input_csv_path}...")
    df = pd.read_csv(input_csv_path)
except FileNotFoundError:
    print(f"❌ Error: Could not find {input_csv_path}.")
    df = pd.DataFrame()

if not df.empty:
    print("=" * 70)
    print(f"🚀 Starting Phi-3 RAG Evaluation over {len(df)} test prompts...")
    print("=" * 70)

    # Initialize CSV Headers for the 10 metric categories
    headers = [
        "Query", "Expected", "Response", "Pages",
        "Exact Match", "Token F1", "Token P", "Token R",
        "BLEU-1", "BLEU-2", "BLEU-4", "ROUGE-1", "ROUGE-2", "ROUGE-L",
        "BERTScore F1", "BERTScore P", "BERTScore R", "Semantic Similarity",
        "Faithfulness", "Answer Relevancy", "Context Recall",
        "Retrieval Time (s)", "Generation Time (s)", "Tokens/sec"
    ]

    with open(output_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()

    for index, row in df.iterrows():
        query = row['Prompt (Question)']
        expected = row['Ground Truth (Expected Answer)']

        print(f"\n🔄 Processing Q{index+1}/{len(df)}: {str(query)[:60]}...")

        try:
            res = process_query(query, expected)

            print(f"\n📝 Answer:\n{res['response']}")
            print("\n📚 Validated Sources:")
            for j, (doc, score) in enumerate(res['scored_docs'][:3]):
                print(f"- Page {doc.metadata.get('page')} (Context Score: {score:.2f})")

            print(f"\n✅ RAG Scores (Fth/Rel/Ctx): {res['faithfulness']},{res['answer_rel']},{res['ctx_recall']} | F1: {res['tok_f1']} | Speed: {res['tokens_per_sec']} T/s")
            print("-" * 60)

            # Log ALL Metrics to CSV
            row_data = {
                "Query": query, "Expected": expected, "Response": res['response'], "Pages": res['source_pages'],
                "Exact Match": res['exact_match'], "Token F1": res['tok_f1'], "Token P": res['tok_p'], "Token R": res['tok_r'],
                "BLEU-1": res['bleu1'], "BLEU-2": res['bleu2'], "BLEU-4": res['bleu4'],
                "ROUGE-1": res['rouge1'], "ROUGE-2": res['rouge2'], "ROUGE-L": res['rougeL'],
                "BERTScore F1": res['bert_f1'], "BERTScore P": res['bert_p'], "BERTScore R": res['bert_r'],
                "Semantic Similarity": res['cos_sim'],
                "Faithfulness": res['faithfulness'], "Answer Relevancy": res['answer_rel'], "Context Recall": res['ctx_recall'],
                "Retrieval Time (s)": res['retrieval_time'], "Generation Time (s)": res['generation_time'], "Tokens/sec": res['tokens_per_sec']
            }

            with open(output_csv_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=headers)
                writer.writerow(row_data)

        except Exception as e:
            print(f"❌ Error on Q{index+1}: {e}")

        # Garbage collection after each query
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n✨ DONE! Comprehensive Evaluation results saved to {output_csv_path}")

📂 Loading /content/2024ESC-compressed.pdf...
Processing page 0/101...
Processing page 10/101...
Processing page 20/101...
Processing page 30/101...
Processing page 40/101...
Processing page 50/101...
Processing page 60/101...
Processing page 70/101...
Processing page 80/101...
Processing page 90/101...
Processing page 100/101...
💾 Saving extracted data to extracted_pdf_data.txt...
✅ Loaded 101 pages and saved extraction to extracted_pdf_data.txt.
⚙️ Initializing MedCPT on cuda...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🧠 Embedding Documents...
🧹 Optimizing Memory: Removing Article Encoder...
⚖️ Loading Cross-Encoder (Reranker)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      [Phi-3 Judge Raw Output]: [10, 10, 10]

📝 Answer:
Permanent AF is defined as AF for which no further attempts at restoration of sinus rhythm are planned, after a shared decision between the patient and physician.

📚 Validated Sources:
- Page 14 (Context Score: 3.29)
- Page 13 (Context Score: 0.56)
- Page 8 (Context Score: 0.25)

✅ RAG Scores (Fth/Rel/Ctx): 10,10,10 | F1: 0.875 | Speed: 4.44 T/s
------------------------------------------------------------

🔄 Processing Q20/20: For patients undergoing cardiac surgery, when is surgical cl...
🔍 Deep Searching...
🤖 Synthesizing Answer...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      [Phi-3 Judge Raw Output]: [10, 10, 10]

📝 Answer:
For patients undergoing cardiac surgery, surgical closure of the left atrial appendage (LAA) is recommended for stroke prevention in patients with atrial fibrillation (AF). This recommendation applies to those who are undergoing cardiac surgery to prevent ischemic stroke and thromboembolism. The procedure is advised as an adjunct to oral anticoagulation in patients with AF.

📚 Validated Sources:
- Page 12 (Context Score: 5.68)
- Page 48 (Context Score: 3.80)
- Page 81 (Context Score: 2.73)

✅ RAG Scores (Fth/Rel/Ctx): 10,10,10 | F1: 0.4286 | Speed: 7.26 T/s
------------------------------------------------------------

✨ DONE! Comprehensive Evaluation results saved to rag_evaluation_phi3_full_metrics.csv
